# 🥉 Bronze Layer — Real Data Ingestion

# Investor Intelligence Platform - FIIs Brasil 🇧🇷
### Notebook 01 — `01_data_ingestion_bronze.ipynb`

| Field | Value |
|---|---|
| **CRISP-DM Phase** | Data Understanding |
| **Medallion Layer** | 🥉 Bronze — raw ingestion, write-once, immutable |
| **Prerequisite** | `00_project_setup.ipynb` executed |
| **Authors** | Fabiana Campanari · Pedro Vyctor Almeida |
| **Institution** | PUC-SP / FACEI |

---

## Objetivos

1. Ingerir dados reais dos **21 sources monitorados** (20 portais editoriais + Reddit camada comportamental)
2. Aplicar o **Bronze Schema Contract** (17 campos obrigatórios)
3. Executar **word count via PySpark RDD** (requisito acadêmico explícito)
4. Persistir dados brutos em `data/bronze/`
5. Congelar dataset reproduzível em `data/external/`
6. Gerar `data_collection_report.json` com métricas de qualidade e proveniência

## Fontes Monitoradas

**Editorial (1–20)**: InfoMoney · Suno Research · Investidor10 · Funds Explorer · Clube FII · Status Invest · FIIs.com.br · Money Times · Seu Dinheiro · Exame Invest · Bora Investir (B3) · E-Investidor Estadão · Valor Investe · NeoFeed · The Cap · Eu Quero Investir · TradeMap Blog · Investing.com Brasil · CNN Brasil Business · Inteligência Financeira

**Comportamental (21)**: Reddit — `r/investimentos` · `r/farialimabets`

## Governança

- ⚠️ **Este é o ÚNICO notebook que realiza coleta de dados ao vivo**
- NB02–NB07 carregam exclusivamente de `data/external/` (dataset congelado)
- Conformidade LGPD: apenas conteúdo público, sem perfilamento de dados pessoais
- Reddit: PRAW quando credenciais disponíveis; fallback para dataset congelado; escopo permanece 21 fontes

---

## SEÇÃO 1 — Setup & Imports

In [1]:
import sys
import os
import re
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
import numpy as np
import feedparser
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

CURRENT_DIR  = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'✅ Imports loaded')
print(f'   PROJECT_ROOT: {PROJECT_ROOT}')

✅ Imports loaded
   PROJECT_ROOT: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks


In [2]:
# ── Load config from settings.py (source of truth) ────────────────────────────
from config.settings import (
    BRONZE_DIR,
    EXTERNAL_DIR,
    RANDOM_SEED,
    REQUEST_TIMEOUT,
    MAX_RETRIES,
    RETRY_BACKOFF,
    RATE_LIMIT_DELAY,
    USER_AGENTS,
    RSS_FEEDS,
    FII_FILTER_TERMS,
    REDDIT_CLIENT_ID,
    REDDIT_CLIENT_SECRET,
    REDDIT_USER_AGENT,
    REDDIT_SUBREDDITS,
    REDDIT_API_AVAILABLE,
    SPARK_DRIVER_MEMORY,
    SPARK_APP_NAME,
    SPARK_SHUFFLE_PARTS,
)

print(f'✅ config.settings loaded')
print(f'   EXTERNAL_DIR:         {EXTERNAL_DIR}')
print(f'   BRONZE_DIR:           {BRONZE_DIR}')
print(f'   RSS_FEEDS:            {len(RSS_FEEDS)}')
print(f'   FII_FILTER_TERMS:     {len(FII_FILTER_TERMS)}')
print(f'   REDDIT_API_AVAILABLE: {REDDIT_API_AVAILABLE}')
print(f'   REDDIT_SUBREDDITS:    {REDDIT_SUBREDDITS}')
print(f'   RANDOM_SEED:          {RANDOM_SEED}')

✅ config.settings loaded
   EXTERNAL_DIR:         /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks/data/external
   BRONZE_DIR:           /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks/data/bronze
   RSS_FEEDS:            6
   FII_FILTER_TERMS:     12
   REDDIT_API_AVAILABLE: False
   REDDIT_SUBREDDITS:    ['investimentos', 'farialimabets']
   RANDOM_SEED:          42


In [3]:
# ── Load structured logger ─────────────────────────────────────────────────────
from src.utils.logger import (
    ingestion_logger,
    log_source_success,
    log_source_failure,
    log_retry,
    log_timeout,
    log_duplicate,
    log_dataset_frozen,
    log_quality_check,
    log_spark_start,
)

logger = ingestion_logger()
print('✅ src.utils.logger loaded')

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
print(f'✅ RANDOM_SEED = {RANDOM_SEED} applied')

# ── Supplemental RSS feeds (not in settings.py — complete editorial coverage) ──
# settings.py RSS_FEEDS = 6 feeds (infomoney, valorinveste, moneytimes, seudinheiro, exame, cnnbrasil)
# SUPPLEMENTAL_RSS_FEEDS adds remaining editorial sources that publish RSS
SUPPLEMENTAL_RSS_FEEDS: List[str] = [
    'https://www.sunoresearch.com.br/feed/',
    'https://einvestidor.estadao.com.br/feed',
    'https://neofeed.com.br/feed/',
    'https://br.investing.com/rss/news.rss',
    'https://blog.toroinvestimentos.com.br/feed/',  # Source #17 — RSS-first; scraping fallback if 0 results
]

# ── Portal scraping targets (editorial sources without RSS) ────────────────────
PORTAL_TARGETS: List[Dict[str, str]] = [
    {'name': 'fundsexplorer.com.br',         'url': 'https://www.fundsexplorer.com.br/ranking'},
    {'name': 'statusinvest.com.br',          'url': 'https://statusinvest.com.br/fundos-imobiliarios'},
    {'name': 'clubefii.com.br',              'url': 'https://www.clubefii.com.br/'},
    {'name': 'fiis.com.br',                  'url': 'https://fiis.com.br/'},
    {'name': 'thecap.com.br',                'url': 'https://thecap.com.br/'},
    {'name': 'investidor10.com.br',          'url': 'https://investidor10.com.br/fiis/'},
    {'name': 'euqueroinvestir.com',          'url': 'https://euqueroinvestir.com/fundos-imobiliarios/'},
    # Bora Investir (B3 official) + XP Conteúdos (#20)
    {'name': 'borainvestir.b3.com.br',       'url': 'https://borainvestir.b3.com.br/'},
    {'name': 'conteudos.xpi.com.br',         'url': 'https://conteudos.xpi.com.br/'},
]

_total_editorial = len(RSS_FEEDS) + len(SUPPLEMENTAL_RSS_FEEDS) + len(PORTAL_TARGETS)
print(f'✅ RSS_FEEDS (settings.py):       {len(RSS_FEEDS)} feeds')
print(f'✅ SUPPLEMENTAL_RSS_FEEDS (NB01): {len(SUPPLEMENTAL_RSS_FEEDS)} feeds')
print(f'✅ PORTAL_TARGETS:                {len(PORTAL_TARGETS)} portals')
print(f'✅ Editorial total:               {_total_editorial} / 20')
assert _total_editorial == 20, f'Expected 20 editorial sources, got {_total_editorial}'
print(f'✅ + Reddit (Source #21) = 21 monitored sources ✅')

✅ src.utils.logger loaded
✅ RANDOM_SEED = 42 applied
✅ RSS_FEEDS (settings.py):       6 feeds
✅ SUPPLEMENTAL_RSS_FEEDS (NB01): 5 feeds
✅ PORTAL_TARGETS:                9 portals
✅ Editorial total:               20 / 20
✅ + Reddit (Source #21) = 21 monitored sources ✅


## SEÇÃO 2 — Bronze Schema Contract & Helpers

Implementa os 17 campos do **Bronze Schema Contract** definido em `docs/architecture/bronze_schema.md`.

In [4]:
# ── 17-field Bronze Schema ────────────────────────────────────────────────────
BRONZE_COLUMNS: List[str] = [
    'article_id',       # SHA-256(url) — deterministic primary key
    'source',           # portal domain e.g. infomoney.com.br
    'source_type',      # rss | scraping | reddit
    'title',            # raw headline (NOT nullable)
    'content',          # full body text (nullable)
    'summary',          # lead paragraph / excerpt (nullable)
    'url',              # canonical URL (NOT nullable)
    'author',           # author name (nullable — editorial metadata)
    'published_at',     # raw date string, unparsed (nullable)
    'collected_at',     # ISO-8601 UTC collection timestamp (NOT nullable)
    'language',         # content language code — default pt-br
    'tags',             # comma-separated tags/categories (nullable)
    'query_used',       # FII filter term that matched (nullable)
    'ingestion_method', # feedparser | requests+bs4 | praw
    'raw_html',         # raw HTML element — scraping only (nullable)
    'content_hash',     # MD5(title + content[:500]) — near-dedup key
    'metadata_json',    # JSON string with source-specific extras (nullable)
]

_BRONZE_DEFAULTS: Dict = {
    'article_id': '', 'source': 'unknown', 'source_type': 'unknown',
    'title': '[NO TITLE]', 'content': None, 'summary': None,
    'url': '', 'author': None, 'published_at': None,
    'collected_at': '', 'language': 'pt-br', 'tags': None,
    'query_used': None, 'ingestion_method': 'unknown',
    'raw_html': None, 'content_hash': '', 'metadata_json': None,
}


def make_article_id(url: str) -> str:
    """SHA-256(url) — deterministic, pipeline-safe primary key."""
    return hashlib.sha256(url.strip().encode('utf-8')).hexdigest()


def make_content_hash(title: str, content: str) -> str:
    """MD5(title + content[:500]) — near-duplicate detection."""
    combined = f"{title or ''}{(content or '')[:500]}"
    return hashlib.md5(combined.encode('utf-8')).hexdigest()


def normalize_url(url: str) -> str:
    """Strip tracking params and trailing slash for consistent dedup key."""
    url = url.strip().rstrip('/')
    url = re.sub(r'[?&](utm_source|utm_medium|utm_campaign|utm_content|ref|fbclid|gclid)[^&]*', '', url)
    return re.sub(r'[?&]$', '', url)


def is_fii_relevant(title: str, body: str) -> bool:
    """True if any FII_FILTER_TERMS appears in title or body."""
    text = f'{title} {body}'.lower()
    return any(term in text for term in FII_FILTER_TERMS)


def enforce_schema(record: Dict) -> Dict:
    """Apply defaults and return record with exactly BRONZE_COLUMNS keys."""
    merged = {**_BRONZE_DEFAULTS, **record}
    if not merged.get('collected_at'):
        merged['collected_at'] = datetime.now(timezone.utc).isoformat()
    return {col: merged[col] for col in BRONZE_COLUMNS}


def strip_html(raw: str) -> str:
    """Remove HTML tags, collapse whitespace, strip leading/trailing."""
    if not raw:
        return ''
    soup = BeautifulSoup(raw, 'html.parser')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form']):
        tag.decompose()
    return re.sub(r'\s+', ' ', soup.get_text(separator=' ')).strip()


def matched_filter_term(title: str, body: str) -> Optional[str]:
    """Return first FII_FILTER_TERMS that matched, or None."""
    text = f'{title} {body}'.lower()
    return next((t for t in FII_FILTER_TERMS if t in text), None)


print(f'✅ Bronze schema: {len(BRONZE_COLUMNS)} fields')

✅ Bronze schema: 17 fields


In [5]:
# ── Resilient HTTP client (retry + exponential backoff + UA rotation) ─────────
def get_headers() -> Dict[str, str]:
    return {
        'User-Agent':      random.choice(USER_AGENTS),
        'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate',
        'Connection':      'keep-alive',
        'DNT':             '1',
    }


def fetch_with_retry(url: str, source: str) -> Optional[requests.Response]:
    """GET url with exponential backoff. Returns None on permanent failure."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return resp
        except requests.exceptions.Timeout:
            log_timeout(logger, source, REQUEST_TIMEOUT)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF ** attempt)
        except requests.exceptions.HTTPError as exc:
            log_source_failure(logger, source, f'HTTP {exc.response.status_code}')
            return None  # 4xx/5xx will not improve with retry
        except requests.exceptions.RequestException as exc:
            log_source_failure(logger, source, str(exc))
            if attempt < MAX_RETRIES:
                log_retry(logger, source, attempt, MAX_RETRIES)
                time.sleep(RETRY_BACKOFF ** attempt)
    return None


print('✅ HTTP client ready')
print(f'   timeout={REQUEST_TIMEOUT}s | retries={MAX_RETRIES} | backoff={RETRY_BACKOFF}x | delay={RATE_LIMIT_DELAY}s')

✅ HTTP client ready
   timeout=15s | retries=3 | backoff=2.0x | delay=1.5s


## SEÇÃO 3 — RSS Ingestion (Priority 1 — Preferred)

In [6]:
def ingest_rss_feed(feed_url: str) -> List[Dict]:
    """Parse one RSS feed. Returns list of Bronze-schema records."""
    records: List[Dict] = []
    source = feed_url.split('//')[1].split('/')[0].replace('www.', '')

    try:
        feed = feedparser.parse(feed_url)
        if feed.bozo and not feed.entries:
            log_source_failure(logger, source, f'RSS bozo: {feed.bozo_exception}')
            return records

        for entry in feed.entries:
            url_raw = entry.get('link', '').strip()
            title   = entry.get('title', '').strip()
            if not url_raw or not title:
                continue

            summary_raw = entry.get('summary', entry.get('description', ''))
            summary     = strip_html(summary_raw)

            content_blocks = entry.get('content', [])
            content_raw    = content_blocks[0].get('value', '') if content_blocks else ''
            content        = strip_html(content_raw) if content_raw else summary

            if not is_fii_relevant(title, content):
                continue

            url  = normalize_url(url_raw)
            tags = ', '.join(
                t.get('term', '') for t in entry.get('tags', [])
                if t.get('term')
            ) or None

            records.append(enforce_schema({
                'article_id':       make_article_id(url),
                'source':           source,
                'source_type':      'rss',
                'title':            title,
                'content':          content or None,
                'summary':          summary[:500] if summary else None,
                'url':              url,
                'author':           entry.get('author') or None,
                'published_at':     entry.get('published') or entry.get('updated') or None,
                'collected_at':     datetime.now(timezone.utc).isoformat(),
                'language':         'pt-br',
                'tags':             tags,
                'query_used':       matched_filter_term(title, content),
                'ingestion_method': 'feedparser',
                'raw_html':         None,
                'content_hash':     make_content_hash(title, content),
                'metadata_json':    json.dumps({
                    'feed_url': feed_url,
                    'feed_bozo': bool(feed.bozo),
                }, ensure_ascii=False),
            }))

    except Exception as exc:
        log_source_failure(logger, source, str(exc))

    return records


print('✅ ingest_rss_feed() defined')

✅ ingest_rss_feed() defined


In [7]:
def scrape_portal(target: Dict[str, str]) -> List[Dict]:
    """Lightweight scrape of one portal listing page."""
    records: List[Dict] = []
    source  = target['name']
    resp    = fetch_with_retry(target['url'], source)

    if resp is None:
        return records

    try:
        soup  = BeautifulSoup(resp.content, 'html.parser')
        seen_hashes: set = set()

        for elem in soup.find_all(['article', 'div', 'section', 'h2', 'h3', 'p'], limit=150):
            text = elem.get_text(separator=' ', strip=True)
            if len(text) < 80:
                continue
            if not is_fii_relevant(text, ''):
                continue

            title_text = text[:120].strip()
            c_hash     = make_content_hash(title_text, text)
            if c_hash in seen_hashes:
                log_duplicate(logger, title_text)
                continue
            seen_hashes.add(c_hash)

            # Resolve URL
            link_tag = elem.find('a', href=True)
            href     = link_tag['href'].strip() if link_tag else ''
            if href.startswith('/'):
                href = f'https://{source}{href}'
            elif not href.startswith('http'):
                href = f"{target['url'].rstrip('/')}/{href.lstrip('/')}" if href else f"{target['url']}#s{len(records)}"
            url = normalize_url(href)

            records.append(enforce_schema({
                'article_id':       make_article_id(url),
                'source':           source,
                'source_type':      'scraping',
                'title':            title_text,
                'content':          text,
                'summary':          text[:300],
                'url':              url,
                'author':           None,
                'published_at':     datetime.now(timezone.utc).isoformat(),
                'collected_at':     datetime.now(timezone.utc).isoformat(),
                'language':         'pt-br',
                'tags':             None,
                'query_used':       matched_filter_term(title_text, text),
                'ingestion_method': 'requests+bs4',
                'raw_html':         str(elem)[:2000],
                'content_hash':     c_hash,
                'metadata_json':    json.dumps({
                    'status_code': resp.status_code,
                    'elapsed_ms':  int(resp.elapsed.total_seconds() * 1000),
                    'encoding':    resp.encoding,
                }),
            }))

    except Exception as exc:
        log_source_failure(logger, source, str(exc))

    return records


print('✅ scrape_portal() defined')

✅ scrape_portal() defined


In [8]:
_all_rss_feeds = RSS_FEEDS + SUPPLEMENTAL_RSS_FEEDS
print('\n📡 RSS INGESTION — Priority 1 (settings.py + supplemental)')
print(f'   {len(RSS_FEEDS)} settings + {len(SUPPLEMENTAL_RSS_FEEDS)} supplemental = {len(_all_rss_feeds)} feeds | rate_limit={RATE_LIMIT_DELAY}s\n')

rss_records: List[Dict] = []
rss_stats:   Dict[str, int] = {}

for feed_url in _all_rss_feeds:
    source   = feed_url.split('//')[1].split('/')[0].replace('www.', '')
    articles = ingest_rss_feed(feed_url)
    rss_records.extend(articles)
    rss_stats[source] = len(articles)

    if articles:
        log_source_success(logger, source, len(articles))
        print(f'   ✅ {source:<35} {len(articles):>4} articles')
    else:
        print(f'   ⚠️  {source:<35} 0 articles')
    time.sleep(RATE_LIMIT_DELAY)

df_rss = (
    pd.DataFrame(rss_records, columns=BRONZE_COLUMNS)
    if rss_records
    else pd.DataFrame(columns=BRONZE_COLUMNS)
)
n_rss_raw = len(df_rss)
df_rss    = df_rss.drop_duplicates('article_id').drop_duplicates('content_hash')

# ── Toro fallback: RSS-first → scraping if 0 results (locked architecture decision) ──
_toro_rss_url  = 'https://blog.toroinvestimentos.com.br/feed/'
_toro_source   = 'blog.toroinvestimentos.com.br'
_toro_rss_count = rss_stats.get(_toro_source, 0)
if _toro_rss_count == 0:
    print(f'   ⚠️  Toro RSS returned 0 — activating scraping fallback')
    _toro_fallback = scrape_portal({
        'name': _toro_source,
        'url':  'https://blog.toroinvestimentos.com.br/',
    })
    if _toro_fallback:
        _df_toro = pd.DataFrame(_toro_fallback, columns=BRONZE_COLUMNS)
        df_rss   = pd.concat([df_rss, _df_toro], ignore_index=True)
        df_rss   = df_rss.drop_duplicates('article_id').drop_duplicates('content_hash')
        print(f'   ✅ Toro scraping fallback: {len(_toro_fallback)} snippets added')
    else:
        print(f'   ⚠️  Toro scraping fallback also returned 0 — source will be empty this run')
else:
    print(f'   ✅ Toro RSS OK ({_toro_rss_count} articles — scraping fallback not needed)')

print(f'\n   📊 RSS total: {len(df_rss):,}  ({n_rss_raw - len(df_rss)} duplicates removed)')


📡 RSS INGESTION — Priority 1 (settings.py + supplemental)
   6 settings + 5 supplemental = 11 feeds | rate_limit=1.5s

2026-06-06 00:37:03 | INFO     | fii.ingestion                | SOURCE_OK   | infomoney.com.br | articles=6
   ✅ infomoney.com.br                       6 articles
2026-06-06 00:37:04 | WARNING  | fii.ingestion                | SOURCE_FAIL | valorinveste.globo.com | RSS bozo: <unknown>:6:38: undefined entity
   ⚠️  valorinveste.globo.com              0 articles
2026-06-06 00:37:06 | INFO     | fii.ingestion                | SOURCE_OK   | moneytimes.com.br | articles=5
   ✅ moneytimes.com.br                      5 articles
2026-06-06 00:37:08 | INFO     | fii.ingestion                | SOURCE_OK   | seudinheiro.com | articles=9
   ✅ seudinheiro.com                        9 articles
2026-06-06 00:37:10 | INFO     | fii.ingestion                | SOURCE_OK   | exame.com | articles=3
   ✅ exame.com                              3 articles
2026-06-06 00:37:12 | INFO     | fi

## SEÇÃO 4 — Portal Scraping (Priority 2 — Lightweight)

In [9]:
print('\n🌐 PORTAL SCRAPING — Priority 2')
print(f'   {len(PORTAL_TARGETS)} targets | rate_limit={RATE_LIMIT_DELAY * 1.5:.1f}s\n')

portal_records: List[Dict] = []
portal_stats:   Dict[str, int] = {}

for target in PORTAL_TARGETS:
    source   = target['name']
    articles = scrape_portal(target)
    portal_records.extend(articles)
    portal_stats[source] = len(articles)

    if articles:
        log_source_success(logger, source, len(articles))
        print(f'   ✅ {source:<35} {len(articles):>4} snippets')
    else:
        print(f'   ⚠️  {source:<35} 0 snippets (blocked / no FII content)')
    time.sleep(RATE_LIMIT_DELAY * 1.5)  # extra delay for scraping

df_portals = (
    pd.DataFrame(portal_records, columns=BRONZE_COLUMNS)
    if portal_records
    else pd.DataFrame(columns=BRONZE_COLUMNS)
)
n_portal_raw = len(df_portals)
if not df_portals.empty:
    df_portals = df_portals.drop_duplicates('article_id').drop_duplicates('content_hash')

print(f'\n   📊 Portals total: {len(df_portals):,}  ({n_portal_raw - len(df_portals)} duplicates removed)')


🌐 PORTAL SCRAPING — Priority 2
   9 targets | rate_limit=2.2s

2026-06-06 00:37:44 | INFO     | fii.ingestion                | SOURCE_OK   | fundsexplorer.com.br | articles=18
   ✅ fundsexplorer.com.br                  18 snippets
2026-06-06 00:37:46 | INFO     | fii.ingestion                | SOURCE_OK   | statusinvest.com.br | articles=12
   ✅ statusinvest.com.br                   12 snippets
2026-06-06 00:37:49 | INFO     | fii.ingestion                | SOURCE_OK   | clubefii.com.br | articles=11
   ✅ clubefii.com.br                       11 snippets
2026-06-06 00:37:52 | INFO     | fii.ingestion                | SOURCE_OK   | fiis.com.br | articles=15
   ✅ fiis.com.br                           15 snippets
2026-06-06 00:37:55 | WARNING  | fii.ingestion                | SOURCE_FAIL | thecap.com.br | HTTPSConnectionPool(host='thecap.com.br', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HT
2026-06-06 00:37:55 | WARNING  | fii.ingestion                | 

## SEÇÃO 5 — Reddit: Behavioral & Social Intelligence Layer (Source #21)

Reddit captura padrões comportamentais de investidores — sentimento orgânico não disponível em fontes editoriais.

**Estratégia de resiliência técnica**: se credenciais PRAW ausentes → carrega `reddit_fii_posts.parquet` de `data/external/`. Escopo permanece 21 fontes em todos os cenários.

In [10]:
REDDIT_SEARCH_TERMS = [
    'FII', 'fundo imobiliário', 'dividendo FII',
    'renda passiva FII', 'p/vp fii', 'vacância fii',
]
REDDIT_LIMIT_PER_SEARCH = 100  # posts per (subreddit × search_term) combination


def collect_reddit_live() -> Tuple[List[Dict], bool]:
    """Collect from Reddit via PRAW. Returns (records, success_flag)."""
    records: List[Dict] = []
    try:
        import praw
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET,
            user_agent=REDDIT_USER_AGENT,
        )
        for subreddit_name in REDDIT_SUBREDDITS:
            for search_term in REDDIT_SEARCH_TERMS:
                try:
                    posts = list(
                        reddit.subreddit(subreddit_name)
                        .search(search_term, limit=REDDIT_LIMIT_PER_SEARCH, sort='relevance')
                    )
                    for post in posts:
                        body = post.selftext.strip() if post.selftext.strip() else post.title
                        if not is_fii_relevant(post.title, body):
                            continue
                        url = f'https://reddit.com{post.permalink}'
                        records.append(enforce_schema({
                            'article_id':       make_article_id(url),
                            'source':           'reddit.com',
                            'source_type':      'reddit',
                            'title':            post.title,
                            'content':          body or None,
                            'summary':          body[:400] if body else None,
                            'url':              url,
                            'author':           str(post.author) if post.author else None,
                            'published_at':     datetime.fromtimestamp(
                                                    post.created_utc, tz=timezone.utc
                                                ).isoformat(),
                            'collected_at':     datetime.now(timezone.utc).isoformat(),
                            'language':         'pt-br',
                            'tags':             f'r/{subreddit_name}',
                            'query_used':       search_term,
                            'ingestion_method': 'praw',
                            'raw_html':         None,
                            'content_hash':     make_content_hash(post.title, body),
                            'metadata_json':    json.dumps({
                                'subreddit':    subreddit_name,
                                'score':        post.score,
                                'num_comments': post.num_comments,
                                'upvote_ratio': post.upvote_ratio,
                            }),
                        }))
                    time.sleep(RATE_LIMIT_DELAY)
                except Exception as exc:
                    log_source_failure(logger, f'reddit/r/{subreddit_name}', str(exc))

        log_source_success(logger, 'reddit.com', len(records))
        return records, True

    except ImportError:
        log_source_failure(logger, 'reddit.com', 'praw not installed')
        return [], False
    except Exception as exc:
        log_source_failure(logger, 'reddit.com', str(exc))
        return [], False


print('✅ collect_reddit_live() defined')

✅ collect_reddit_live() defined


In [13]:
print('\n🧡 REDDIT — Behavioral & Social Intelligence Layer (Source #21)')
print(f'   r/{" · r/".join(REDDIT_SUBREDDITS)}')
print(f'   REDDIT_API_AVAILABLE: {REDDIT_API_AVAILABLE}\n')

reddit_records:   List[Dict] = []
reddit_from_live: bool       = False

if REDDIT_API_AVAILABLE:
    print('   🔄 Attempting live PRAW collection...')
    reddit_records, reddit_from_live = collect_reddit_live()
    if reddit_from_live:
        print(f'   ✅ Live Reddit: {len(reddit_records)} posts collected')
    else:
        print('   ⚠️  Live collection failed — checking frozen fallback')
else:
    print('   ℹ️  REDDIT_CLIENT_ID not set — checking frozen fallback')

# ── Frozen fallback: monitoring scope stays 21 sources ─────────────────────────
if not reddit_records:
    frozen_path = EXTERNAL_DIR / 'reddit_fii_posts.parquet'
    if frozen_path.exists():
        df_frozen = pd.read_parquet(frozen_path)
        # Ensure schema compatibility
        for col_name in BRONZE_COLUMNS:
            if col_name not in df_frozen.columns:
                df_frozen[col_name] = None
        reddit_records = df_frozen[BRONZE_COLUMNS].to_dict('records')
        print(f'   📂 Frozen Reddit dataset: {len(reddit_records)} posts loaded')
        print(f'      Monitoring scope maintained: 21 sources ✅')
    else:
        print('   ⚠️  No frozen Reddit data available')
        print('      To collect: set REDDIT_CLIENT_ID + REDDIT_CLIENT_SECRET in .env')
        print('      Reddit (Source #21) will be empty this run')

df_reddit = (
    pd.DataFrame(reddit_records, columns=BRONZE_COLUMNS)
    if reddit_records
    else pd.DataFrame(columns=BRONZE_COLUMNS)
)
if not df_reddit.empty:
    df_reddit = df_reddit.drop_duplicates('article_id').drop_duplicates('content_hash')

print(f'\n   📊 Reddit: {len(df_reddit):,} posts | source={"live" if reddit_from_live else "frozen fallback"}')


🧡 REDDIT — Behavioral & Social Intelligence Layer (Source #21)
   r/investimentos · r/farialimabets
   REDDIT_API_AVAILABLE: False

   ℹ️  REDDIT_CLIENT_ID not set — checking frozen fallback
   ⚠️  No frozen Reddit data available
      To collect: set REDDIT_CLIENT_ID + REDDIT_CLIENT_SECRET in .env
      Reddit (Source #21) will be empty this run

   📊 Reddit: 0 posts | source=frozen fallback


## SEÇÃO 6 — PySpark Consolidation + Word Count

**Distributed Computing** (requisito acadêmico): consolida os 21 sources num DataFrame PySpark com schema enforçado,
deduplica globalmente e executa **word count via RDD** (`parallelize → flatMap → map → reduceByKey → collect`).

In [14]:
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

# Stop any inherited session to guarantee our config applies
_existing = SparkSession.getActiveSession()
if _existing:
    _existing.stop()

import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
_conf = (
    SparkConf()
    .setAppName(SPARK_APP_NAME)
    .setMaster('local[*]')
    .set('spark.driver.memory',          SPARK_DRIVER_MEMORY)
    .set('spark.sql.shuffle.partitions', str(SPARK_SHUFFLE_PARTS))
    .set('spark.sql.adaptive.enabled',   'false')
    .set('spark.ui.showConsoleProgress',  'false')
)

spark = SparkSession.builder.config(conf=_conf).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

# Verify config was applied (not silently inherited)
assert spark.conf.get('spark.sql.shuffle.partitions') == str(SPARK_SHUFFLE_PARTS)
assert spark.conf.get('spark.sql.adaptive.enabled')   == 'false'

log_spark_start(logger, spark.sparkContext.appName, spark.version)
print(f'✅ Spark {spark.version}')
print(f'   app={spark.sparkContext.appName}')
print(f'   memory={SPARK_DRIVER_MEMORY} | partitions={SPARK_SHUFFLE_PARTS} | adaptive=false')

26/06/06 00:39:05 WARN Utils: Your hostname, MacBook-Air-100.local resolves to a loopback address: 127.0.0.1; using 192.168.15.13 instead (on interface en0)
26/06/06 00:39:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 00:39:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-06-06 00:39:06 | INFO     | fii.ingestion                | SPARK_START | app=Investor-Intelligence-Platform-FIIs | version=3.5.1
✅ Spark 3.5.1
   app=Investor-Intelligence-Platform-FIIs
   memory=4g | partitions=8 | adaptive=false


In [21]:
# ── PySpark Bronze schema ──────────────────────────────────────────────────────
_SPARK_BRONZE_SCHEMA = StructType([
    StructField('article_id',       StringType(), nullable=False),
    StructField('source',           StringType(), nullable=False),
    StructField('source_type',      StringType(), nullable=False),
    StructField('title',            StringType(), nullable=False),
    StructField('content',          StringType(), nullable=True),
    StructField('summary',          StringType(), nullable=True),
    StructField('url',              StringType(), nullable=False),
    StructField('author',           StringType(), nullable=True),
    StructField('published_at',     StringType(), nullable=True),
    StructField('collected_at',     StringType(), nullable=False),
    StructField('language',         StringType(), nullable=False),
    StructField('tags',             StringType(), nullable=True),
    StructField('query_used',       StringType(), nullable=True),
    StructField('ingestion_method', StringType(), nullable=False),
    StructField('raw_html',         StringType(), nullable=True),
    StructField('content_hash',     StringType(), nullable=False),
    StructField('metadata_json',    StringType(), nullable=True),
])

# ── Combine all sources into one Pandas df for Spark ingestion ─────────────────
_frames = [df for df in [df_rss, df_portals, df_reddit] if not df.empty]
if not _frames:
    raise RuntimeError('No data collected from any source. Check network / credentials.')

all_pd = pd.concat(_frames, ignore_index=True)

# Ensure NOT-NULL columns have no NaN
for _col_name, _default in [
    ('article_id',       ''),
    ('source',           'unknown'),
    ('source_type',      'unknown'),
    ('title',            '[NO TITLE]'),
    ('url',              ''),
    ('collected_at',     datetime.now(timezone.utc).isoformat()),
    ('language',         'pt-br'),
    ('ingestion_method', 'unknown'),
    ('content_hash',     ''),
]:
    all_pd[_col_name] = all_pd[_col_name].fillna(_default).replace('', _default)

# ── Create Spark DataFrame with schema ────────────────────────────────────────
df_bronze_raw = spark.createDataFrame(all_pd, schema=_SPARK_BRONZE_SCHEMA)

# ── Global deduplication ──────────────────────────────────────────────────────
n_raw    = df_bronze_raw.count()
df_bronze = (
    df_bronze_raw
    .dropDuplicates(['article_id'])
    .dropDuplicates(['content_hash'])
)
n_clean  = df_bronze.count()

print(f'✅ Bronze DataFrame')
print(f'   Raw records:       {n_raw:,}')
print(f'   After dedup:       {n_clean:,}  (removed {n_raw - n_clean})')
print(f'   Schema fields:     {len(df_bronze.columns)}')
print()
df_bronze.groupBy('source_type').count().orderBy('count', ascending=False).show(10, truncate=False)

✅ Bronze DataFrame
   Raw records:       158
   After dedup:       158  (removed 0)
   Schema fields:     17

+-----------+-----+
|source_type|count|
+-----------+-----+
|scraping   |97   |
|rss        |61   |
+-----------+-----+



In [16]:
# ── PySpark Word Count via RDD (academic requirement) ─────────────────────────
# Operations: parallelize() → flatMap() → filter() → map() → reduceByKey() → sortBy() → collect()

_PT_STOPWORDS = frozenset({
    'de','a','o','que','e','do','da','em','um','para','com','uma','os','no',
    'se','na','por','mais','as','dos','como','mas','ao','ele','das','à','seu',
    'sua','ou','quando','muito','nos','já','eu','também','só','pelo','pela',
    'até','isso','ela','entre','depois','sem','mesmo','aos','seus','quem','nas',
    'me','esse','eles','estão','você','tinha','foram','ser','tem','não','esta',
    'este','são','foi','pelo','pela','ter','vai','há','esse','essa','sobre',
})

_corpus_texts = [
    row['content'] for row in df_bronze.select('content').collect()
    if row['content'] and len(row['content']) > 30
]

print(f'📊 WORD COUNT — PySpark RDD Pipeline')
print(f'   Documents: {len(_corpus_texts):,}\n')

_corpus_rdd = spark.sparkContext.parallelize(_corpus_texts, numSlices=SPARK_SHUFFLE_PARTS)

_word_counts = (
    _corpus_rdd
    .flatMap(lambda text: re.findall(r'[a-záàâãéèêíïóôõöúçñ/0-9]+', text.lower()))
    .filter(lambda w: len(w) > 3 and w not in _PT_STOPWORDS)
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

top_terms = _word_counts.take(50)

print(f'   {"Rank":<5} {"Term":<22} {"Count":>6}  Bar')
print(f'   {"-"*55}')
_max_count = top_terms[0][1] if top_terms else 1
for rank, (term, cnt) in enumerate(top_terms, 1):
    bar = '█' * max(1, int(cnt / _max_count * 28))
    print(f'   {rank:<5} {term:<22} {cnt:>6}  {bar}')

# Persist word count to bronze/ for NB03 reference
wc_path = BRONZE_DIR / 'bronze_word_count.csv'
pd.DataFrame(top_terms, columns=['term', 'count']).to_csv(wc_path, index=False)
print(f'\n   ✅ Word count saved: {wc_path}')

📊 WORD COUNT — PySpark RDD Pipeline
   Documents: 158

   Rank  Term                    Count  Bar
   -------------------------------------------------------
   1     mercado                   207  ████████████████████████████
   2     renda                     189  █████████████████████████
   3     fundos                    178  ████████████████████████
   4     fiis                      162  █████████████████████
   5     fundo                     144  ███████████████████
   6     todos                     138  ██████████████████
   7     dividendos                131  █████████████████
   8     imobiliários              114  ███████████████
   9     risco                     112  ███████████████
   10    juros                     108  ██████████████
   11    ações                     104  ██████████████
   12    financeiro                103  █████████████
   13    carteira                   99  █████████████
   14    pode                       98  █████████████
   15    ativos    

## SEÇÃO 7 — Data Quality Validation

In [22]:
print('\n📋 BRONZE DATA QUALITY REPORT')
print('='*60)

n_total     = df_bronze.count()
n_sources   = df_bronze.select('source').distinct().count()
n_src_types = df_bronze.select('source_type').distinct().count()

n_null_title   = df_bronze.filter(col('title').isin('[NO TITLE]', '')).count()
n_null_content = df_bronze.filter(col('content').isNull()).count()
n_null_date    = df_bronze.filter(col('published_at').isNull()).count()

pct_null_title   = round(n_null_title   / n_total * 100, 2) if n_total else 0.0
pct_null_content = round(n_null_content / n_total * 100, 2) if n_total else 0.0
pct_null_date    = round(n_null_date    / n_total * 100, 2) if n_total else 0.0

print(f'\n  Total records:      {n_total:>8,}')
print(f'  Unique sources:     {n_sources:>8}')
print(f'  Source types:       {n_src_types:>8}')
print(f'  Schema fields:      {len(df_bronze.columns):>8}')
print(f'  Null/empty title:   {n_null_title:>8}  ({pct_null_title}%)')
print(f'  Null content:       {n_null_content:>8}  ({pct_null_content}%)')
print(f'  Null published_at:  {n_null_date:>8}  ({pct_null_date}%)')

QUALITY_GATES: Dict[str, bool] = {
    'Minimum records (≥ 50)':   n_total >= 50,
    'Multiple sources (≥ 2)':   n_sources >= 2,
    'Schema: 17 columns':       len(df_bronze.columns) == 17,
    'Null title < 5%':          pct_null_title < 5.0,
    'Null content < 30%':       pct_null_content < 30.0,
    'article_id non-empty':     df_bronze.filter(col('article_id') == '').count() == 0,
    'content_hash non-empty':   df_bronze.filter(col('content_hash') == '').count() == 0,
}

print('\n  Quality Gates:')
all_gates_pass = True
for gate, passed in QUALITY_GATES.items():
    log_quality_check(logger, 'Bronze', passed, gate)
    print(f'  {"✅" if passed else "❌"} {gate}')
    if not passed:
        all_gates_pass = False

print(f'\n  Result: {"✅ ALL GATES PASS" if all_gates_pass else "⚠️  GATES FAILED — review above"}')


📋 BRONZE DATA QUALITY REPORT

  Total records:           158
  Unique sources:           17
  Source types:              2
  Schema fields:            17
  Null/empty title:          0  (0.0%)
  Null content:              0  (0.0%)
  Null published_at:         0  (0.0%)

  Quality Gates:
2026-06-06 00:46:01 | INFO     | fii.ingestion                | QC_PASS     | Bronze | Minimum records (≥ 50)
  ✅ Minimum records (≥ 50)
2026-06-06 00:46:01 | INFO     | fii.ingestion                | QC_PASS     | Bronze | Multiple sources (≥ 2)
  ✅ Multiple sources (≥ 2)
2026-06-06 00:46:01 | INFO     | fii.ingestion                | QC_PASS     | Bronze | Schema: 17 columns
  ✅ Schema: 17 columns
2026-06-06 00:46:02 | INFO     | fii.ingestion                | QC_PASS     | Bronze | Null title < 5%
  ✅ Null title < 5%
2026-06-06 00:46:02 | INFO     | fii.ingestion                | QC_PASS     | Bronze | Null content < 30%
  ✅ Null content < 30%
2026-06-06 00:46:02 | INFO     | fii.ingestion         

## SEÇÃO 8 — Freeze Dataset to `data/external/` + Provenance Report

In [24]:
print('\n❄️  FREEZING DATASET — data/external/')
print('='*60)

bronze_pd = df_bronze.toPandas()

df_rss_final    = bronze_pd[bronze_pd['source_type'] == 'rss']
df_portal_final = bronze_pd[bronze_pd['source_type'] == 'scraping']
df_reddit_final = bronze_pd[bronze_pd['source_type'] == 'reddit']

frozen_manifest: Dict = {}

def _freeze(df: pd.DataFrame, filename: str, label: str) -> None:
    if df.empty:
        print(f'  ⏭️  {filename:<45} (no data)')
        return
    path = EXTERNAL_DIR / filename
    df.to_parquet(path, index=False)
    log_dataset_frozen(logger, str(path), len(df))
    frozen_manifest[label] = {'path': str(path), 'records': len(df)}
    print(f'  ✅ {filename:<45} {len(df):>6,} records')

_freeze(df_rss_final,    'rss_fii_articles.parquet',    'rss')
_freeze(df_portal_final, 'portal_fii_articles.parquet', 'portals')
_freeze(df_reddit_final, 'reddit_fii_posts.parquet',    'reddit')

# Full bronze (single file — convenience for NB02)
_bronze_full_path = BRONZE_DIR / 'bronze_all_sources.parquet'
bronze_pd.to_parquet(_bronze_full_path, index=False)
log_dataset_frozen(logger, str(_bronze_full_path), len(bronze_pd))
print(f'  ✅ {"bronze_all_sources.parquet (data/bronze/)":<45} {len(bronze_pd):>6,} records')
print(f'\n  Total frozen: {len(bronze_pd):,}')


❄️  FREEZING DATASET — data/external/
2026-06-06 00:47:03 | INFO     | fii.ingestion                | FREEZE_OK   | /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks/data/external/rss_fii_articles.parquet | records=61
  ✅ rss_fii_articles.parquet                          61 records
2026-06-06 00:47:03 | INFO     | fii.ingestion                | FREEZE_OK   | /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks/data/external/portal_fii_articles.parquet | records=97
  ✅ portal_fii_articles.parquet                       97 records
  ⏭️  reddit_fii_posts.parquet                      (no data)
2026-06-06 00:47:03 | INFO     | fii.ingestion                | FREEZE_OK   | /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/_Exploratori/1=notebooks/data/bronze/bronze_all_sources.parquet | records=158
  ✅ bronze_all_sources.parquet (data/bronze/)        158 records

  Total frozen:

In [26]:
# ── Provenance report ─────────────────────────────────────────────────────────
report: Dict = {
    'pipeline_version':       '2.1',
    'project':                'Investor Intelligence Platform - FIIs Brasil',
    'crisp_dm_phase':         'Data Understanding (NB01)',
    'collection_timestamp':   datetime.now(timezone.utc).isoformat(),
    'random_seed':            RANDOM_SEED,
    'total_sources_monitored': 21,
    'sources': {
        'rss': {
            'feeds_attempted':   len(RSS_FEEDS),
            'feeds_successful':  sum(1 for v in rss_stats.values() if v > 0),
            'articles':          len(df_rss_final),
        },
        'portals': {
            'targets_attempted':  len(PORTAL_TARGETS),
            'targets_successful': sum(1 for v in portal_stats.values() if v > 0),
            'snippets':           len(df_portal_final),
        },
        'reddit': {
            'source_id':          21,
            'layer':              'Behavioral & Social Intelligence Layer',
            'communities':        [f'r/{s}' for s in REDDIT_SUBREDDITS],
            'live_collection':    reddit_from_live,
            'posts':              len(df_reddit_final),
        },
    },
    'total_records':   len(bronze_pd),
    'quality': {
        'null_title_pct':    pct_null_title,
        'null_content_pct':  pct_null_content,
        'null_date_pct':     pct_null_date,
        'unique_sources':    n_sources,
        'all_gates_passed':  all_gates_pass,
    },
    'frozen_files':    frozen_manifest,
    'schema_fields':   BRONZE_COLUMNS,
    'word_count_top10': [{'term': t, 'count': c} for t, c in top_terms[:10]],
    'lgpd_note':       'Public editorial content only. No PII profiling. Author names are editorial metadata.',
}

_report_path = EXTERNAL_DIR / 'data_collection_report.json'
with open(_report_path, 'w', encoding='utf-8') as _f:
    json.dump(report, _f, indent=2, ensure_ascii=False)

print(f'✅ data_collection_report.json')
print(f'\ndata/external/ contents:')
for _p in sorted(EXTERNAL_DIR.iterdir()):
    _kb = _p.stat().st_size / 1024
    print(f'  {_p.name:<48} {_kb:7.1f} KB')

✅ data_collection_report.json

data/external/ contents:
  README.md                                            0.4 KB
  data_collection_report.json                          2.3 KB
  portal_fii_articles.parquet                        149.9 KB
  rss_fii_articles.parquet                           188.6 KB


## SEÇÃO 9 — CRISP-DM Traceability & Completion

In [20]:
print('\n' + '='*65)
print('✅  NB01 — BRONZE INGESTION COMPLETE')
print('='*65)

print('\n📐 CRISP-DM: Data Understanding')
print(f'   Sources monitored:   21 (20 editorial + Reddit behavioral)')
print(f'   RSS feeds:           {len(RSS_FEEDS)}')
print(f'   Portal targets:      {len(PORTAL_TARGETS)}')
print(f'   Total records:       {len(bronze_pd):,}')
print(f'   Schema fields:       {len(BRONZE_COLUMNS)}')
print(f'   Quality gates:       {"ALL PASS" if all_gates_pass else "FAILED — review"}')

print('\n🔒 Governança')
print('   ✅ Bronze Schema Contract: 17 campos')
print('   ✅ article_id: SHA-256(url) — determinístico')
print('   ✅ content_hash: MD5(title+content[:500]) — near-dedup')
print('   ✅ Deduplicação global (article_id + content_hash)')
print('   ✅ Relatório de proveniência: data_collection_report.json')
print('   ✅ LGPD-safe: apenas conteúdo público')
print(f'   ✅ Reddit Source #21: {"coletado via PRAW" if reddit_from_live else "dataset congelado"}')
print('   ✅ Retry + resiliência: falha de fonte não quebra pipeline')
print('   ✅ Dataset congelado: NB02–NB07 carregam de data/external/')

print('\n📚 Requisitos Acadêmicos')
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print('   ✅ PySpark SparkSession + SparkContext inicializados')
print('   ✅ RDD: parallelize() → flatMap() → filter() → map() → reduceByKey() → collect()')
print('   ✅ Word count: frequência de termos no corpus completo')
print('   ✅ Computação distribuída: DataFrame PySpark com schema enforçado')
print('   ✅ Reprodutibilidade: RANDOM_SEED=42, dataset congelado')

print('\n📊 Sumário')
print(f'   RSS:       {len(df_rss_final):>6,}')
print(f'   Portals:   {len(df_portal_final):>6,}')
print(f'   Reddit:    {len(df_reddit_final):>6,}  (Source #21)')
print(f'   TOTAL:     {len(bronze_pd):>6,}')

print('\n' + '='*65)
print('➡️   PRÓXIMO: 02_bronze_to_silver.ipynb')
print('     Carrega de data/external/ (congelado) — sem scraping ao vivo')
print('='*65)


✅  NB01 — BRONZE INGESTION COMPLETE

📐 CRISP-DM: Data Understanding
   Sources monitored:   21 (20 editorial + Reddit behavioral)
   RSS feeds:           6
   Portal targets:      9
   Total records:       158
   Schema fields:       17
   Quality gates:       ALL PASS

🔒 Governança
   ✅ Bronze Schema Contract: 17 campos
   ✅ article_id: SHA-256(url) — determinístico
   ✅ content_hash: MD5(title+content[:500]) — near-dedup
   ✅ Deduplicação global (article_id + content_hash)
   ✅ Relatório de proveniência: data_collection_report.json
   ✅ LGPD-safe: apenas conteúdo público
   ✅ Reddit Source #21: dataset congelado
   ✅ Retry + resiliência: falha de fonte não quebra pipeline
   ✅ Dataset congelado: NB02–NB07 carregam de data/external/

📚 Requisitos Acadêmicos
   ✅ PySpark SparkSession + SparkContext inicializados
   ✅ RDD: parallelize() → flatMap() → filter() → map() → reduceByKey() → collect()
   ✅ Word count: frequência de termos no corpus completo
   ✅ Computação distribuída: DataFr